# Regularization: Ridge, Lasso & ElasticNet
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ajit-ai/Data_Science/blob/main/03_Machine_Learning/algorithms/ridge_lasso_regularization.ipynb)

When models get too flexible they fit noise. Regularization penalizes large coefficients: Ridge (L2) shrinks them smoothly, Lasso (L1) drives some exactly to zero = automatic feature selection. ElasticNet blends both.

**Covered:** polynomial overfitting demo, coefficient paths, alpha selection via CV.

## 1. Create an overfit-prone regression problem

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import cross_val_score

rng = np.random.RandomState(42)
X = np.sort(rng.uniform(0, 1, 40)).reshape(-1, 1)
y = np.sin(6 * X).ravel() + rng.normal(0, 0.25, 40)

for deg in [1, 3, 15]:
    model = make_pipeline(PolynomialFeatures(deg), LinearRegression())
    err = -cross_val_score(model, X, y, scoring="neg_mean_squared_error", cv=5).mean()
    print(f"degree {deg:2d}: CV MSE = {err:.4f}")

In [ ]:
xs = np.linspace(0, 1, 300).reshape(-1, 1)
plt.scatter(X, y, s=18, label="data")
for deg, style in [(1, "--"), (3, "-"), (15, ":")]:
    m = make_pipeline(PolynomialFeatures(deg), LinearRegression()).fit(X, y)
    plt.plot(xs, m.predict(xs), style, label=f"degree {deg}")
plt.ylim(-2, 2); plt.legend(); plt.title("Degree 15 chases every noise point"); plt.show()

## 2. Ridge vs Lasso on the degree-15 model

In [ ]:
models = {
    "OLS       ": make_pipeline(PolynomialFeatures(15), LinearRegression()),
    "Ridge a=.1": make_pipeline(PolynomialFeatures(15), StandardScaler(), Ridge(alpha=0.1)),
    "Lasso a=.01": make_pipeline(PolynomialFeatures(15), StandardScaler(), Lasso(alpha=0.01)),
}
for name, m in models.items():
    err = -cross_val_score(m, X, y, scoring="neg_mean_squared_error", cv=5).mean()
    print(f"{name} CV MSE = {err:.4f}")

Note: regularization must be applied on **scaled** features, hence the `StandardScaler` inside the pipeline.

## 3. Lasso performs feature selection (zero coefficients)

In [ ]:
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split

Xh, yh = fetch_california_housing(return_X_y=True, as_frame=False)
Xtr, Xte, ytr, yte = train_test_split(Xh, yh, random_state=42)
sc = StandardScaler().fit(Xtr); Xtr_s, Xte_s = sc.transform(Xtr), sc.transform(Xte)

las = Lasso(alpha=0.02).fit(Xtr_s, ytr)
names = fetch_california_housing().feature_names
print("kept features :", [(n, round(w, 3)) for n, w in zip(names, las.coef_) if w != 0])
print("dropped       :", [n for n, w in zip(names, las.coef_) if w == 0])
print(f"R2 test = {las.score(Xte_s, yte):.4f}")

## 4. Choosing alpha with validation curves

In [ ]:
alphas = np.logspace(-4, 1, 30)
ridge_cv = [ -cross_val_score(make_pipeline(StandardScaler(), Ridge(a)), Xh, yh,
             scoring="neg_mean_squared_error").mean() for a in alphas]
plt.loglog(alphas, ridge_cv, "o-")
plt.xlabel("alpha"); plt.ylabel("CV MSE"); plt.title("Pick alpha at the valley"); plt.show()

## Rules of thumb
- Default first try: `Ridge` (stable, handles correlated features).
- Want a smaller feature set? `Lasso` (or ElasticNet when features correlate in groups).
- Tune alpha on a log grid with CV - never on the test set.